# Models — does review text help predict rating?

Sibling of `01_models_retail.ipynb`, but the target is **`rating`** instead of
`retail`. `retail` flips from target to predictor; rows are still filtered to the
2nd–90th retail percentile ($11–$80), here as a data filter.

Flow mirrors `01`: **baseline → target encoding → tuning → final CV**, each run with
and without `retail`. The headline flips too: for price, text didn't help — for
rating it does, and the **best block is `basic + keywords`**.

## Methods to improve accuracy

Same upgrades as `01` (target-encode categoricals, tune + early-stop XGBoost, honest
held-out test). Unlike price, **review keywords genuinely help rating** — so the
final / CV / importance / SHAP models here use **`basic + keywords`**, not `basic`
alone.

**Next:** hyper-parameter search (Optuna), LightGBM / CatBoost.

In [19]:
import numpy as np
import pandas as pd
import itables
from itables import show
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

itables.options.columnDefs = [{"className": "dt-left", "targets": "_all"}]

FEATURES_BASIC_PATH =   r"..\..\features\features_basic.parquet"
# KEYWORDS_PATH = r"..\..\features\features_keywords.parquet"
KEYWORDS_PATH =         r"..\..\features\features_keywords_robust.parquet"
EMBEDDINGS_PCA_PATH =   r"..\..\features\features_embeddings_PCA.parquet"
ANCHORED_PATH =         r"..\..\features\features_embeddings_anchored.parquet"
FULL_EMBEDDINGS_PATH =  r"..\..\features\features_embeddings.parquet"


## Load features

In [20]:
features = pd.read_parquet(FEATURES_BASIC_PATH)
keywords = pd.read_parquet(KEYWORDS_PATH)
embeddings_pca = pd.read_parquet(EMBEDDINGS_PCA_PATH)
embeddings_anchored = pd.read_parquet(ANCHORED_PATH)
full_embeddings = pd.read_parquet(FULL_EMBEDDINGS_PATH)
print(f"number of features in each data model:\n  features: {features.shape}  \n  keywords: {keywords.shape}  \n  "
      f"emb_pca: {embeddings_pca.shape}  \n  anchored: {embeddings_anchored.shape}  \n  full: {full_embeddings.shape}")

target = "rating"

basic_features = [
    "retail", "alcohol", "bottle_size", "vintage", "case_production",
    "country_ord", "wine_type_ord", "state_ord", "company_ord",
    "appellation_ord", "varietal_label_ord", "age_at_review",
]
basic_features_excl_retail = [f for f in basic_features if f != "retail"]

kw_features     = [c for c in keywords.columns if c.startswith("kw_") and not c.endswith("_count")]
pca_features    = [c for c in embeddings_pca.columns if c.startswith("pca_")]
anchor_features = [c for c in embeddings_anchored.columns if c.startswith("anchor_")]
full_features   = [c for c in full_embeddings.columns if c.startswith("emb_")]

df = (
    features
    .merge(keywords[["wine_id"] + kw_features], on="wine_id", how="left")
    .merge(embeddings_pca[["wine_id"] + pca_features], on="wine_id", how="left")
    .merge(embeddings_anchored[["wine_id"] + anchor_features], on="wine_id", how="left")
    .merge(full_embeddings[["wine_id"] + full_features], on="wine_id", how="left")
)
assert len(df) == len(features), "merge changed row count — wine_id not unique?"
print(f"Full table merged: {df.shape}")
print(f"basic features ({len(basic_features)}): {basic_features}")
print(f"aroma features ({len(kw_features)}): {kw_features}")
print(f"emb-PCA features ({len(pca_features)}): {pca_features}")
print(f"anchor features ({len(anchor_features)}): {anchor_features}")
print(f"full embeddings features ({len(full_features)}): {full_features}")


number of features in each data model:
  features: (135192, 15)  
  keywords: (135192, 107)  
  emb_pca: (135192, 21)  
  anchored: (135192, 23)  
  full: (135192, 385)
Full table merged: (135192, 494)
basic features (12): ['retail', 'alcohol', 'bottle_size', 'vintage', 'case_production', 'country_ord', 'wine_type_ord', 'state_ord', 'company_ord', 'appellation_ord', 'varietal_label_ord', 'age_at_review']
aroma features (53): ['kw_dry', 'kw_acidic', 'kw_tart', 'kw_sweet', 'kw_caramel', 'kw_alcohol', 'kw_strong', 'kw_balanced', 'kw_citrus', 'kw_apple', 'kw_pear', 'kw_strawberry', 'kw_raspberry', 'kw_cherry', 'kw_red', 'kw_black', 'kw_blackberry', 'kw_tropical', 'kw_banana', 'kw_pineapple', 'kw_lichi', 'kw_stone', 'kw_peach', 'kw_soil', 'kw_mineral', 'kw_oak', 'kw_coconut', 'kw_vanilla', 'kw_smoke', 'kw_tannic', 'kw_light', 'kw_heavy', 'kw_body', 'kw_flowers', 'kw_floral', 'kw_grass', 'kw_herbs', 'kw_spicy', 'kw_vegetables', 'kw_pepper', 'kw_bell', 'kw_earth', 'kw_leather', 'kw_tea', 'kw_

## Baseline — default XGB, simple split

Out-of-the-box `XGBRegressor()` on an 80/20 split, ordinal encoding, no tuning —
the reference the tuned pipeline below is measured against.

In [21]:
# Default out-of-the-box XGBoost with a plain 80/20 train-test split (no val set,
# no target encoding) — same simple recipe as 01_models_retail.

model_df = df[basic_features + kw_features + pca_features + anchor_features + full_features + [target]].dropna(subset=[target])
low, high = model_df["retail"].quantile([0.02, 0.90])
model_df = model_df[model_df["retail"].between(low, high)]
print(f"Kept retail ${low:.2f} - ${high:.2f}  ({len(model_df):,} rows)  | target = {target}")


# train and evaluate a model given a list of features — XGBRegressor with defaults made explicit
def train_eval_basic(feature_list):
    X, y = model_df[feature_list], model_df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = xgb.XGBRegressor(
        # core boosting
        n_estimators=100,
        learning_rate=0.3,
        max_depth=6,
        min_child_weight=1,
        gamma=0,
        # sampling
        subsample=1.0,
        colsample_bytree=1.0,
        # regularization
        reg_lambda=1.0,
        reg_alpha=0,
        # objective / method
        objective="reg:squarederror",
        booster="gbtree",
        tree_method="auto",
        # misc
        n_jobs=-1,
        random_state=42,
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    return model, {
        "n_features": len(feature_list),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "MAE":  mean_absolute_error(y_test, pred),
        "R2":   r2_score(y_test, pred),
    }



Kept retail $11.00 - $80.00  (113,220 rows)  | target = rating


### Compare feature blocks (with `retail`)

In [22]:
# default-XGB results across the same four feature blocks (with retail)
_, b1 = train_eval_basic(basic_features)
_, b2 = train_eval_basic(basic_features + kw_features)
_, b3 = train_eval_basic(basic_features + pca_features)
_, b4 = train_eval_basic(basic_features + anchor_features)
_, b5 = train_eval_basic(basic_features + full_features)

results_default = pd.DataFrame([
    {"model": "1: basic",            **b1},
    {"model": "2: basic + kw",       **b2},
    {"model": "3: basic + emb-PCA",  **b3},
    {"model": "4: basic + anchors",  **b4},
    {"model": "5: basic + full_embeddings",  **b5},
])
results_default["dR2_vs_base"] = results_default["R2"] - b1["R2"]
results_default.round(4)

,model,n_features,RMSE,MAE,R2,dR2_vs_base
0,1: basic,12,1.8317,1.4381,0.4517,0.0000
1,2: basic + kw,65,1.6684,1.3079,0.5451,0.0934
2,3: basic + emb-PCA,32,1.7875,1.3992,0.4778,0.0261
3,4: basic + anchors,34,1.8193,1.4301,0.4591,0.0074
4,5: basic + full_embeddings,396,1.6327,1.2785,0.5644,0.1127


### Baseline — 5-fold CV

Variance check on the default-XGB baseline for the chosen `basic + keywords` block
(train+val; the 15% test stays held out).

In [23]:
from sklearn.model_selection import KFold

train_df, tmp_df = train_test_split(model_df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, random_state=42)
print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

cv_features = basic_features + kw_features  # best block for rating
cv_df = pd.concat([train_df, val_df])          # everything except the held-out test set
X_all, y_all = cv_df[cv_features], cv_df[target] # type: ignore

# Default XGBRegressor — only random_state / n_jobs fixed for reproducibility.
cv_params = dict(random_state=42, n_jobs=-1)

scores = []
for fold, (tr_idx, te_idx) in enumerate(KFold(5, shuffle=True, random_state=42).split(X_all), 1):
    X_tr, X_te = X_all.iloc[tr_idx], X_all.iloc[te_idx]
    y_tr, y_te = y_all.iloc[tr_idx], y_all.iloc[te_idx]
    m = xgb.XGBRegressor(**cv_params).fit(X_tr, y_tr)
    s = r2_score(y_te, m.predict(X_te))
    scores.append(s)
    print(f"fold {fold}: R2={s:.4f}")

scores = np.array(scores)
print(f"\n5-fold CV R2: {scores.mean():.4f} +/- {scores.std():.4f}")

train=79,254  val=16,983  test=16,983
fold 1: R2=0.5379
fold 2: R2=0.5493
fold 3: R2=0.5552
fold 4: R2=0.5410
fold 5: R2=0.5418

5-fold CV R2: 0.5450 +/- 0.0063


## Fine-tuning — target encoding + hyperparameters

70/15/15 train/val/test, shared across models so any difference comes from the
feature block. Two leakage-safe upgrades over the baseline: **target-encode** the
categoricals (fit on train) and **tune** the XGBoost params (early-stopped on val).
Metrics are on the held-out test set. The weak `basic + full_embeddings` block is
dropped from here on.

### Step 1 — target encoding only (default XGB)

In [24]:
# One model_df with every feature block, same rows for all models.
from sklearn.preprocessing import TargetEncoder

# Train / validation / test split (70 / 15 / 15). The 15% TEST set is held out
# and only used for the final metrics reported below; VALIDATION drives XGBoost
# early stopping (and would drive any hyper-parameter search). Same split for
# every model, so metric differences come from the feature block, not the rows.
train_df, tmp_df = train_test_split(model_df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, random_state=42)
print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

# The structured categoricals are ordinal-encoded with an arbitrary integer
# order (misleading for trees). Re-encode them with a CV-smoothed target mean,
# fit on TRAIN ONLY so no validation/test information leaks in.
CAT_ORD = ["country_ord", "wine_type_ord", "state_ord",
           "company_ord", "appellation_ord", "varietal_label_ord"]


def train_eval(feature_list):
    """Fit on train (early-stopped on val), report metrics on the held-out TEST set."""
    cats = [c for c in CAT_ORD if c in feature_list]
    X_train = train_df[feature_list].copy()
    X_val = val_df[feature_list].copy()
    X_test = test_df[feature_list].copy()
    y_train, y_val, y_test = train_df[target], val_df[target], test_df[target]

    if cats:  # leakage-safe target encoding: fit on train, apply to val/test
        enc = TargetEncoder(target_type="continuous", random_state=42)
        X_train[cats] = enc.fit_transform(train_df[cats], y_train)
        X_val[cats] = enc.transform(val_df[cats])
        X_test[cats] = enc.transform(test_df[cats])

    # model = make_model()
    model = xgb.XGBRegressor()

    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    pred = model.predict(X_test)
    return model, {
        "n_features": len(feature_list),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "MAE": mean_absolute_error(y_test, pred),
        "R2": r2_score(y_test, pred),
    }

train=79,254  val=16,983  test=16,983


In [25]:
# train and evaluate models with different feature sets, including rating
model1, m1 = train_eval(basic_features)
model2, m2 = train_eval(basic_features + kw_features)
model3, m3 = train_eval(basic_features + pca_features)
model4, m4 = train_eval(basic_features + anchor_features)

results = pd.DataFrame([
    {"model": "1: basic",            **m1},
    {"model": "2: basic + kw",       **m2},
    {"model": "3: basic + emb-PCA",  **m3},
    {"model": "4: basic + anchors",  **m4},
])
results["dR2_vs_base"] = results["R2"] - m1["R2"]
results.round(4)

,model,n_features,RMSE,MAE,R2,dR2_vs_base
0,1: basic,12,1.7772,1.3965,0.4840,0.0000
1,2: basic + kw,65,1.6094,1.2624,0.5769,0.0928
2,3: basic + emb-PCA,32,1.7081,1.3410,0.5234,0.0393
3,4: basic + anchors,34,1.7463,1.3737,0.5018,0.0178


### Step 2 — target encoding + tuned XGB

In [26]:
# One model_df with every feature block, same rows for all models.
from sklearn.preprocessing import TargetEncoder

# Train / validation / test split (70 / 15 / 15). The 15% TEST set is held out
# and only used for the final metrics reported below; VALIDATION drives XGBoost
# early stopping (and would drive any hyper-parameter search). Same split for
# every model, so metric differences come from the feature block, not the rows.
train_df, tmp_df = train_test_split(model_df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, random_state=42)
print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

# The structured categoricals are ordinal-encoded with an arbitrary integer
# order (misleading for trees). Re-encode them with a CV-smoothed target mean,
# fit on TRAIN ONLY so no validation/test information leaks in.
CAT_ORD = ["country_ord", "wine_type_ord", "state_ord",
           "company_ord", "appellation_ord", "varietal_label_ord"]


# Change the XGBRegressor parameters - at hand tuning.
def make_model():
    """Tuned, early-stopped XGBoost (replaces the previous default XGBRegressor())."""
    return xgb.XGBRegressor(
        n_estimators=900,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_lambda=2.0,
        early_stopping_rounds=80,
        eval_metric="rmse",
        random_state=42,
        n_jobs=-1,
    )


def train_eval(feature_list, params=None):
    """Fit on train (early-stopped on val), report metrics on the held-out TEST set.

    `params=None` -> use make_model() (early-stopped). Pass a params dict (e.g.
    `cv_params`, which has no early_stopping_rounds) to train that config instead.
    """
    cats = [c for c in CAT_ORD if c in feature_list]
    X_train = train_df[feature_list].copy()
    X_val = val_df[feature_list].copy()
    X_test = test_df[feature_list].copy()
    y_train, y_val, y_test = train_df[target], val_df[target], test_df[target]

    if cats:  # leakage-safe target encoding: fit on train, apply to val/test
        enc = TargetEncoder(target_type="continuous", random_state=42)
        X_train[cats] = enc.fit_transform(train_df[cats], y_train)
        X_val[cats] = enc.transform(val_df[cats])
        X_test[cats] = enc.transform(test_df[cats])

    model = make_model() if params is None else xgb.XGBRegressor(**params)

    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    pred = model.predict(X_test)
    return model, {
        "n_features": len(feature_list),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "MAE": mean_absolute_error(y_test, pred),
        "R2": r2_score(y_test, pred),
    }

train=79,254  val=16,983  test=16,983


In [27]:
# train and evaluate models with different feature sets, including rating
model1, m1 = train_eval(basic_features)
model2, m2 = train_eval(basic_features + kw_features)
model3, m3 = train_eval(basic_features + pca_features)
model4, m4 = train_eval(basic_features + anchor_features)

results = pd.DataFrame([
    {"model": "1: basic",            **m1},
    {"model": "2: basic + kw",       **m2},
    {"model": "3: basic + emb-PCA",  **m3},
    {"model": "4: basic + anchors",  **m4},
])
results["dR2_vs_base"] = results["R2"] - m1["R2"]
results.round(4)

,model,n_features,RMSE,MAE,R2,dR2_vs_base
0,1: basic,12,1.7457,1.3700,0.5022,0.0000
1,2: basic + kw,65,1.5584,1.2211,0.6032,0.1011
2,3: basic + emb-PCA,32,1.6553,1.2976,0.5524,0.0502
3,4: basic + anchors,34,1.6959,1.3297,0.5302,0.0280


### 5-fold CV — final config

Variance estimate for the final model (`basic + keywords` + target encoding + tuned
XGB) on train+val; encoding re-fit inside each fold. Test set untouched.

In [28]:
from sklearn.model_selection import KFold

cv_features = basic_features + kw_features  # best block for rating
cv_df = pd.concat([train_df, val_df])          # train + val; the 15% test set stays untouched
X_all, y_all = cv_df[cv_features], cv_df[target]
cats = [c for c in CAT_ORD if c in cv_features]

# Final tuned config. There is no early-stopping fold inside CV, so n_estimators
# is fixed near the early-stopped optimum found above.
cv_params = dict(
    n_estimators=900,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=2.0,
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1,
)

# 5-fold CV: each fold trains on 4/5 of the rows and is scored on the held-out 1/5.
# Target encoding is re-fit inside every fold (on that fold's training rows only),
# so no information leaks from the held-out rows.
fold_scores = []
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_num, (train_idx, holdout_idx) in enumerate(kfold.split(X_all), start=1):
    # this fold's training rows vs its held-out (scored) rows
    X_train_fold, X_holdout_fold = X_all.iloc[train_idx].copy(), X_all.iloc[holdout_idx].copy()
    y_train_fold, y_holdout_fold = y_all.iloc[train_idx], y_all.iloc[holdout_idx]

    # leakage-safe target encoding: fit on the training rows, apply to both
    encoder = TargetEncoder(target_type="continuous", random_state=42)
    X_train_fold[cats] = encoder.fit_transform(X_train_fold[cats], y_train_fold)
    X_holdout_fold[cats] = encoder.transform(X_holdout_fold[cats])

    # train on the training rows, score R² on the held-out rows
    model_fold = xgb.XGBRegressor(**cv_params).fit(X_train_fold, y_train_fold)
    fold_r2 = r2_score(y_holdout_fold, model_fold.predict(X_holdout_fold))
    fold_scores.append(fold_r2)
    print(f"fold {fold_num}: R2={fold_r2:.4f}")

fold_scores = np.array(fold_scores)
print(f"\n5-fold CV R2: {fold_scores.mean():.4f} +/- {fold_scores.std():.4f}")

fold 1: R2=0.6011
fold 2: R2=0.6092
fold 3: R2=0.6148
fold 4: R2=0.6022
fold 5: R2=0.6015

5-fold CV R2: 0.6058 +/- 0.0054


# Rating prediction without `retail`

`retail` is the strongest structured predictor — drop it to see how much rating
leans on price, and whether keywords compensate. Mirrors the steps above.

### Baseline — default XGB

In [29]:
# Same models, but with `retail` dropped from the feature set.
# retail is the strongest structured predictor of rating, so this shows how much
# rating prediction leans on price — and whether the text blocks recover any
# of the lost signal when retail is unavailable. (Models 5-8 mirror 1-4.)
model1, m1 = train_eval_basic(basic_features_excl_retail)
model2, m2 = train_eval_basic(basic_features_excl_retail + kw_features)
model3, m3 = train_eval_basic(basic_features_excl_retail + pca_features)
model4, m4 = train_eval_basic(basic_features_excl_retail + anchor_features)

results_excl = pd.DataFrame([
    {"model": "1: no-retail base",       **m1},
    {"model": "2: no-retail + kw",       **m2},
    {"model": "3: no-retail + emb-PCA",  **m3},
    {"model": "4: no-retail + anchors",  **m4},
])
results_excl["dR2_vs_base"] = results_excl["R2"] - m1["R2"]
results_excl.round(4)

,model,n_features,RMSE,MAE,R2,dR2_vs_base
0,1: no-retail base,11,1.9725,1.5586,0.3641,0.0000
1,2: no-retail + kw,64,1.7814,1.4020,0.4814,0.1173
2,3: no-retail + emb-PCA,31,1.9357,1.5250,0.3876,0.0235
3,4: no-retail + anchors,33,1.9666,1.5537,0.3679,0.0038


### Target encoding + tuned XGB

In [30]:
# Same models, but with `retail` dropped from the feature set.
# retail is the strongest structured predictor of rating, so this shows how much
# rating prediction leans on price — and whether the text blocks recover any
# of the lost signal when retail is unavailable. (Models 5-8 mirror 1-4.)
model5, m5 = train_eval(basic_features_excl_retail)
model6, m6 = train_eval(basic_features_excl_retail + kw_features)
model7, m7 = train_eval(basic_features_excl_retail + pca_features)
model8, m8 = train_eval(basic_features_excl_retail + anchor_features)

results_excl = pd.DataFrame([
    {"model": "5: no-retail base",       **m5},
    {"model": "6: no-retail + kw",       **m6},
    {"model": "7: no-retail + emb-PCA",  **m7},
    {"model": "8: no-retail + anchors",  **m8},
])
results_excl["dR2_vs_base"] = results_excl["R2"] - m5["R2"]
results_excl.round(4)

,model,n_features,RMSE,MAE,R2,dR2_vs_base
0,5: no-retail base,11,1.8363,1.4471,0.4492,0.0000
1,6: no-retail + kw,64,1.6244,1.2772,0.5689,0.1198
2,7: no-retail + emb-PCA,31,1.7342,1.3646,0.5087,0.0595
3,8: no-retail + anchors,33,1.7809,1.4031,0.4819,0.0327


# Feature Importance

## Feature importance — final model (with `retail`)

Best block (`basic + keywords`), tuned config without early stopping
(`params=cv_params`), test R² reported. Importance split by base vs kw block.

In [31]:
# Final model = basic + keywords (the best block for rating), tuned config without
# early stopping (params=cv_params). Reports held-out test R2 + importances.
final_features = basic_features + kw_features
final_model, final_metrics = train_eval(final_features, params=cv_params)
print(f"Final model (with retail) — test R2={final_metrics['R2']:.4f}  "
      f"RMSE={final_metrics['RMSE']:.3f}  MAE={final_metrics['MAE']:.3f}")

imp = (
    pd.DataFrame({"feature": final_features, "importance": final_model.feature_importances_})
    .assign(group=lambda d: np.where(d["feature"].str.startswith("kw_"), "kw", "base"))
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
    .assign(importance=lambda d: d["importance"].round(3))
)
print("total importance by block:")
print(imp.groupby("group")["importance"].sum().round(3).to_string())
show(imp)

Final model (with retail) — test R2=0.6032  RMSE=1.558  MAE=1.221
total importance by block:
group
base    0.317
kw      0.687


Loading ITables v2.7.3 from the internet... (need help?)


## Feature importance — final model (without `retail`)

Same config, `retail` dropped.

In [32]:
# Same best block but WITHOUT `retail`: tuned config without early stopping.
final_features_nr = basic_features_excl_retail + kw_features
final_model_nr, final_metrics_nr = train_eval(final_features_nr, params=cv_params)
print(f"Final model (no retail) — test R2={final_metrics_nr['R2']:.4f}  "
      f"RMSE={final_metrics_nr['RMSE']:.3f}  MAE={final_metrics_nr['MAE']:.3f}")

imp = (
    pd.DataFrame({"feature": final_features_nr, "importance": final_model_nr.feature_importances_})
    .assign(group=lambda d: np.where(d["feature"].str.startswith("kw_"), "kw", "base"))
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
    .assign(importance=lambda d: d["importance"].round(3))
)
print("total importance by block:")
print(imp.groupby("group")["importance"].sum().round(3).to_string())
show(imp)

Final model (no retail) — test R2=0.5689  RMSE=1.624  MAE=1.277
total importance by block:
group
base    0.252
kw      0.748


Loading ITables v2.7.3 from the internet... (need help?)


# SHAP — signed effect in rating points (the "slope" analog)

Tree-SHAP splits each prediction additively, **in rating points**. Per feature:
`mean_abs_pts` = average contribution size (importance in points); `signed_pts` =
that magnitude signed by direction (**+** raises the score, **−** lowers it). Exact
tree SHAP via XGBoost's `pred_contribs`, on a test-set sample.

In [33]:
def shap_signed_points(model, feature_list, explain_df, n_sample=4000):
    """Per-feature SHAP summary in rating points: mean |contribution| and signed direction.

    Uses XGBoost's exact tree SHAP (`pred_contribs`). Categoricals are re-encoded
    with the same train-fitted TargetEncoder the model was trained on.
    """
    cats = [c for c in CAT_ORD if c in feature_list]
    sample = explain_df.sample(min(n_sample, len(explain_df)), random_state=42)
    X = sample[feature_list].copy()
    if cats:
        enc = TargetEncoder(target_type="continuous", random_state=42)
        enc.fit(train_df[cats], train_df[target])
        X[cats] = enc.transform(sample[cats])

    contribs = model.get_booster().predict(xgb.DMatrix(X), pred_contribs=True)[:, :-1]  # drop bias col
    mean_abs = np.abs(contribs).mean(axis=0)
    # direction: sign of feature <-> contribution correlation (pandas .corr drops NaN pairs)
    sign = [np.sign(X[f].reset_index(drop=True).corr(pd.Series(contribs[:, j])))
            for j, f in enumerate(feature_list)]
    return (
        pd.DataFrame({"feature": feature_list,
                      "mean_abs_pts": mean_abs.round(3),
                      "signed_pts": (np.array(sign) * mean_abs).round(3)})
        .sort_values("mean_abs_pts", ascending=False)
        .reset_index(drop=True)
    )

In [34]:
print("Final model WITH retail — signed SHAP effect on rating (points), top 15:")
show(shap_signed_points(final_model, final_features, test_df).head(15))

Final model WITH retail — signed SHAP effect on rating (points), top 15:


Loading ITables v2.7.3 from the internet... (need help?)


In [35]:
print("Final model WITHOUT retail — signed SHAP effect on rating (points), top 15:")
show(shap_signed_points(final_model_nr, final_features_nr, test_df).head(15))

Final model WITHOUT retail — signed SHAP effect on rating (points), top 15:


Loading ITables v2.7.3 from the internet... (need help?)


# Conclusions

## Best model — `basic + keywords`

Unlike price, **review keywords clearly help rating**. Test R² by block (tuned XGB +
target encoding):

| block | with `retail` | without `retail` |
|---|---|---|
| basic | 0.502 | 0.449 |
| **basic + keywords** | **0.603** | **0.569** |
| basic + emb-PCA | 0.552 | 0.509 |
| basic + anchors | 0.530 | 0.482 |

`+kw` adds **+0.10 R²** (with `retail`) / **+0.12** (without), and cheap lexical
keywords beat both embedding blocks — the mirror image of the price model, where
review text only hurt.

## Gradual improvement (best block)

`basic + keywords`, held-out test R²:

| stage | test R² |
|---|---|
| Default XGB, ordinal encoding | 0.545 |
| + Target encoding | 0.577 |
| + Hyperparameter tuning | 0.603 |
| **5-fold CV** (final) | **0.606 ± 0.005** |

## With vs without `retail`

Best model: **0.603 → 0.569** — dropping price costs ~0.03 R² (basic alone loses more,
0.502 → 0.449). Rating leans on price, but keywords recover most of the gap.

## What drives rating — feature importance + SHAP

Gain importance: the **kw block carries ~68%** of it (75% without `retail`), spread
thinly across 53 descriptors. But SHAP (in rating points) shows the biggest *single*
movers are structured: `retail` +0.78, `company` +0.42, `appellation` +0.24; the top
keyword is small (`kw_tannic` −0.08). So many weak keyword cues together lift R²,
while a few structured features make the largest individual swings. Without `retail`,
`company` (+0.59) and `appellation` (+0.47) become the largest movers.

## Takeaway

**Text helps `rating` but not `retail`** (`01`) — a clean asymmetry: prose tracks the
score (same author), price tracks region / producer / tier. Keep `basic + keywords`
for rating; embeddings and anchors add noise over plain word presence.